# 02 Maximum Likelihood

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)
[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)
[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)


### Zero to Hero: Master the Concept
**Concept:** Loops

**What is it?**
A control flow statement for iterating over a sequence (like a list or time periods).

**Why does it matter in Economics?**
Used to simulate an economy over time (`for t in range(T):`) or to iterate over agents.

**Key Takeaway:**
Loops are intuitive but can be slow in Python. For mathematical operations on large arrays, prefer "vectorization" with NumPy.



### Zero to Hero: Master the Concept
**Concept:** Numerical Optimization (Minimization)

**What is it?**
Finding the input value that results in the lowest possible output of a function.

**Why does it matter in Economics?**
Rational agents minimize costs. Econometricians minimize the sum of squared errors (OLS) or negative likelihood.

**Key Takeaway:**
Optimization is at the heart of structural estimation and agent decision problems.



### Code Walkthrough
The following code block implements the logic described above. Here is a step-by-step breakdown:


In [ ]:
# === Environment Setup ===
import os, sys, math, time, random, json, textwrap, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy.optimize import minimize
from scipy.stats import norm, chi2
import statsmodels.api as sm
from IPython.display import Image, display, Markdown

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (11, 7), 'figure.dpi': 130})
np.set_printoptions(suppress=True, linewidth=120, precision=4)

print("Environment initialized for Maximum Likelihood Estimation.")

# The Lens

**What economic problem are we solving?**
In economics, we rarely observe the true data-generating process. We see prices, quantities, and decisions, but we don't see the underlying parameters (preferences, elasticities) that drove them. **Maximum Likelihood Estimation (MLE)** is the detective work of finding the parameters that were *most likely* to have produced the data we see.

**Why do we need this method?**
While Ordinary Least Squares (OLS) is great for linear models, many economic phenomena are non-linear. Binary choices (buy/don't buy), count data (number of patents), and censored data (wages > 0) cannot be correctly estimated with OLS. MLE provides a unified, consistent, and efficient framework for estimating models of almost any form.

## Chapter 6.2: Maximum Likelihood Estimation

---

### Table of Contents

1.  [**Introduction: The Principle of Maximum Likelihood**](#intro)
2.  [**The Likelihood and Log-Likelihood Functions**](#likelihood)
3.  [**The Geometry of the Log-Likelihood Function**](#geometry)
    - [The Score Vector](#score)
    - [The Fisher Information Matrix](#info)
4.  [**Numerical Optimization and Implementation**](#numerical)
    - [A Reusable `MLEstimator` Class](#mle-class)
    - [Visualizing the Log-Likelihood Surface](#surface)
5.  [**Hypothesis Testing: The Holy Trinity**](#trinity)
6.  [**Application: Probit Model for Binary Choice**](#probit)
7.  [**Summary and Key Takeaways**](#summary)


### Zero to Hero: Master the Concept
**Concept:** Python Classes (Object-Oriented Programming)

**What is it?**
A blueprint for creating objects. Classes bundle data (attributes, like *wealth*) and behavior (methods, like *consume*) together.

**Why does it matter in Economics?**
Economic agents (Households, Firms) are naturally modeled as objects. This allows us to create thousands of heterogeneous agents from a single template.

**Key Takeaway:**
Classes are the backbone of Agent-Based Models (ABM) and HANK models.



### Zero to Hero: Master the Concept
**Concept:** Functions

**What is it?**
Reusable blocks of code that perform a specific task. They take inputs (arguments) and return outputs.

**Why does it matter in Economics?**
In economics, functions represent mathematical relationships: `production(k, l)`, `utility(c)`, or `policy_rule(state)`.

**Key Takeaway:**
Writing clean functions makes your model modular and testable.



### Zero to Hero: Master the Concept
**Concept:** Pandas DataFrame

**What is it?**
A 2-dimensional labeled data structure, like a spreadsheet or SQL table.

**Why does it matter in Economics?**
The standard container for economic data (e.g., time series of GDP, cross-sectional survey data).

**Key Takeaway:**
DataFrames allow for powerful manipulation (slicing, filtering, aggregating) with aligned indices.



### Code Walkthrough
The following code block implements the logic described above. Here is a step-by-step breakdown:


In [ ]:
class MLEstimator:
    """
    A class to perform Maximum Likelihood Estimation for a given model.

    This class is designed to be a general-purpose tool for estimating parameters
    of any model for which a log-likelihood function can be specified.
    """

    def __init__(self, loglike_func, data, param_names=None):
        """
        Initializes the MLEstimator.

        Parameters
        ----------
        loglike_func : callable
            The log-likelihood function. Must take two arguments: `params` (a
            NumPy array of parameters) and `data` (the data used for estimation).
            It should return the total log-likelihood value.
        data : object
            The data to be used in estimation. The format is flexible and should
            be handled by the user-provided loglike_func.
        param_names : list of str, optional
            A list of names for the parameters being estimated. If None, generic
            names like 'theta_0', 'theta_1', etc., will be used.
        """
        self.loglike = loglike_func
        self.data = data
        self.param_names = param_names
        self.results = None

    def fit(self, start_params):
        """
        Fit the model using a numerical optimizer to find the MLE.

        Parameters
        ----------
        start_params : np.ndarray
            An array of starting values for the optimization. The length must
            match the number of parameters.

        Returns
        -------
        self
            Returns the instance of the estimator.
        """
        if self.param_names is None:
            self.param_names = [f"theta_{i}" for i in range(len(start_params))]

        # The objective function is the *negative* of the log-likelihood,
        # because scipy.optimize performs minimization.
        def objective(params):
            return -self.loglike(params, self.data)

        # Use the BFGS algorithm to find the minimum of the negative log-likelihood
        # BFGS approximates the Hessian, which we invert to get variance
        res = minimize(objective, start_params, method="BFGS", options={"disp": False}) # Find parameters that minimize the objective

        # Store results
        self.mle_params = res.x
        # The inverse of the Hessian matrix is a consistent estimator of the
        # variance-covariance matrix of the parameters.
        self.vcov = res.hess_inv
        self.std_errs = np.sqrt(np.diag(self.vcov))
        self.loglike_val = -res.fun
        self.results = res
        return self

    def summary(self):
        """
        Display a summary table of the estimation results, similar to those
        produced by standard econometric software.
        """
        if self.results is None:
            print("Model has not been fitted yet.")
            return

        # Calculate z-scores and p-values for hypothesis tests
        z_scores = self.mle_params / self.std_errs
        p_values = norm.sf(np.abs(z_scores)) * 2

        # Calculate 95% confidence intervals
        ci_lower = self.mle_params - 1.96 * self.std_errs
        ci_upper = self.mle_params + 1.96 * self.std_errs

        # Create a pandas DataFrame for a nicely formatted table
        summary_df = pd.DataFrame(
            {
                "Coefficient": self.mle_params,
                "Std. Error": self.std_errs,
                "z-score": z_scores,
                "p-value": p_values,
                "[0.025": ci_lower,
                "0.975]": ci_upper,
            },
            index=self.param_names,
        )

        print(f"Maximum Log-Likelihood: {self.loglike_val:.4f}")
        try:
            # Try to infer N from data structure
            if isinstance(self.data, dict):
                N = len(list(self.data.values())[0])
            else:
                N = len(self.data)
            print(f"Number of Observations: {N}")
        except:
            pass

        display(summary_df.round(4))
        return summary_df


<a id='intro'></a>
## 1. Introduction: The Principle of Maximum Likelihood

**Maximum Likelihood Estimation (MLE)** is the gold standard for parametric estimation. It asks a simple but powerful question:

> *Given the data we have observed, what values of the parameters would make this data the **most probable**?*

Assume we have data $\mathbf{y} = (y_1, ..., y_n)$ drawn from a probability distribution $f(y; \theta)$.
The **Likelihood Function** $L(\theta | \mathbf{y})$ is simply the joint probability of the data, viewed as a function of $\theta$:
$$ L(\theta | \mathbf{y}) = \prod_{i=1}^n f(y_i; \theta) $$

We maximize the **Log-Likelihood** $\mathcal{L}(\theta)$ because sums are easier to differentiate than products:
$$ \mathcal{L}(\theta) = \ln L(\theta) = \sum_{i=1}^n \ln f(y_i; \theta) $$

### Example: MLE for a Bernoulli Process (Coin Flip)

Suppose we flip a coin 10 times and get 8 heads. What is the probability $p$ of heads?
The likelihood is $L(p) = p^8 (1-p)^2$. The log-likelihood is $8 \ln(p) + 2 \ln(1-p)$.
Let's visualize this function.


### Zero to Hero: Master the Concept
**Concept:** Lambda Functions

**What is it?**
Small, anonymous, one-line functions. Syntax: `lambda arguments: expression`.

**Why does it matter in Economics?**
Useful for defining short economic functions on the fly, like a specific utility function `u = lambda c: np.log(c)` to pass into an optimizer.

**Key Takeaway:**
Use them for brevity, but prefer standard `def` functions for complex logic.



### Code Walkthrough
The following code block implements the logic described above. Here is a step-by-step breakdown:


In [ ]:
# Simple example: MLE for the probability 'p' of a Bernoulli trial
n_heads = 8
n_tails = 2
log_likelihood = lambda p: n_heads * np.log(p) + n_tails * np.log(1-p)

p_grid = np.linspace(0.01, 0.99, 200)
ll_vals = log_likelihood(p_grid)
mle_p = n_heads / (n_heads + n_tails)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(p_grid, ll_vals, lw=2.5, label='Log-Likelihood $\mathcal{L}(p)$')
ax.axvline(mle_p, color='r', ls='--', label=f'MLE $\hat{{p}}={mle_p:.2f}$')

# Illustrate the Score (slope) at a test point
p_test = 0.5
ll_test = log_likelihood(p_test)
score_test = (n_heads/p_test) - (n_tails/(1-p_test))
ax.plot(p_test, ll_test, 'go', ms=10)
ax.plot(p_grid, ll_test + score_test * (p_grid - p_test), 'g--', label='Score (Slope) at p=0.5')

ax.set_title('Log-Likelihood Function for a Bernoulli Process')
ax.set_xlabel('Parameter p (Probability of Success)')
ax.set_ylabel('Log-Likelihood Value')
ax.set_ylim(-30, -4)
ax.legend(); plt.show() # Display the generated figure


<a id='numerical'></a>
## 4. Numerical Optimization and Implementation

For most models (like Probit or Logit), we cannot find $\hat{\theta}$ analytically. We use numerical optimization (like the Newton-Raphson or BFGS algorithms) to climb the hill of the log-likelihood function.

We will now demonstrate MLE on a **Probit Model**. A Probit model assumes that there is a latent variable $y^* = X\beta + \epsilon$, where $\epsilon \sim N(0, 1)$. We observe $y=1$ if $y^* > 0$, and $y=0$ otherwise.

The probability of success is:
$$ P(y=1|X) = P(X\beta + \epsilon > 0) = P(\epsilon > -X\beta) = \Phi(X\beta) $$
where $\Phi$ is the standard normal CDF.

The log-likelihood contribution for observation $i$ is:
$$ \mathcal{L}_i(\beta) = y_i \ln \Phi(X_i\beta) + (1-y_i) \ln (1-\Phi(X_i\beta)) $$


### Zero to Hero: Master the Concept
**Concept:** NumPy Arrays

**What is it?**
A grid of values, all of the same type, indexed by a tuple of nonnegative integers.

**Why does it matter in Economics?**
The foundation of numerical computing. Vectors (1D arrays) and matrices (2D arrays) represent economic variables and linear systems.

**Key Takeaway:**
NumPy arrays support "vectorized" operations (e.g., `wage * labor`), which are much faster than looping.



### Code Walkthrough
The following code block implements the logic described above. Here is a step-by-step breakdown:


In [ ]:
# 1. Generate Synthetic Data for Probit
rng = np.random.default_rng(seed=42)
N = 1000
X = sm.add_constant(rng.normal(0, 1, size=(N, 2))) # Constant, x1, x2
true_beta = np.array([-0.5, 1.2, -0.8]) # True parameters

# Latent variable y* = X*beta + e
latent_y = X @ true_beta + rng.normal(size=N) # Matrix multiplication
y = (latent_y > 0).astype(int)

# 2. Define Log-Likelihood for Probit
def neg_loglike_probit(beta, data):
    y, X = data['y'], data['X']
    # Linear predictor
    z = X @ beta # Matrix multiplication
    # Probability (CDF)
    p = norm.cdf(z)
    # Clip probabilities to avoid log(0) errors
    p = np.clip(p, 1e-10, 1 - 1e-10)
    
    # Log-likelihood
    ll = np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return -ll # Minimize negative LL

# 3. Estimate
data_probit = {'y': y, 'X': X}
mle_probit = MLEstimator(neg_loglike_probit, data_probit, param_names=['Const', 'Beta1', 'Beta2'])
mle_probit.fit(start_params=[0, 0, 0])

print("Estimated Probit Model (Manual MLE):")
mle_probit.summary()

### Verification with Statsmodels
Let's compare our manual implementation with the professional `statsmodels` library to ensure correctness.

In [ ]:
sm_model = sm.Probit(y, X)
sm_results = sm_model.fit(disp=0)
print(sm_results.summary())

## 5. Hypothesis Testing: The Holy Trinity

We can visualize the three classical tests (Wald, LR, LM) on the log-likelihood surface. We will test the hypothesis $H_0: \beta_1 = 0$.


### Code Walkthrough
The following code block implements the logic described above. Here is a step-by-step breakdown:


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Create grid for Beta1 vs Beta2 (holding Const fixed at MLE)
b_const = mle_probit.mle_params[0]
b1_vals = np.linspace(0.5, 1.9, 50)
b2_vals = np.linspace(-1.5, -0.1, 50)
B1, B2 = np.meshgrid(b1_vals, b2_vals)
LL = np.zeros_like(B1)

for i in range(50):
    for j in range(50):
        # Calculate LL at this point
        # Note: neg_loglike returns POSITIVE cost, so LL is negative of that
        LL[i, j] = -neg_loglike_probit([b_const, B1[i,j], B2[i,j]], data_probit)

# Contour plot
cs = ax.contour(B1, B2, LL, levels=20, cmap='viridis')
ax.clabel(cs, inline=1, fontsize=10)

# Mark MLE
ax.plot(mle_probit.mle_params[1], mle_probit.mle_params[2], 'r*', ms=15, label='Unrestricted MLE')

# Mark Restricted MLE (where Beta1 = 0)
# We'd normally estimate this formally, but for viz we assume it lies on the axis
ax.axvline(0, color='k', linestyle='--', label='Restriction $\beta_1=0$')

ax.set_title('Log-Likelihood Surface: $\mathcal{L}(\beta_1, \beta_2)$')
ax.set_xlabel('Beta 1')
ax.set_ylabel('Beta 2')
ax.legend()
plt.show() # Display the generated figure


# Summary

1.  **Likelihood Principle**: We estimate parameters by finding the values that maximize the probability of observing the data we actually saw.
2.  **Implementation**: We built a `MLEstimator` class that uses `scipy.optimize` to minimize the negative log-likelihood.
3.  **Flexibility**: This same class solved a Normal distribution estimation and a Probit regression. It can be applied to *any* model where you can write down the log-likelihood (e.g., Poisson, Tobit, GARCH).
4.  **Properties**: MLE is consistent, asymptotically normal, and efficient, making it the default choice for most econometric models.